# Can we predict failure within 90 days of an inspection?
Target: `failed_90d`. Follows `checklist.md`.

Design decisions (made before any modelling):
- Every `asset_tag` appears once, so there are no repeated entities. The model would be used on *future* inspections, so the split is **temporal**: the latest 20% of rows are the test set; CV inside training uses expanding-window time splits.
- `parts_replaced` is recorded on the work order that *closed* the inspection record, i.e. after the inspection, and it is a near-perfect proxy for the outcome (0 parts -> 0% failures, 2+ parts -> 100%). It is **excluded** as leakage. `asset_tag` is an identifier and is dropped; `inspected_at` is used only for ordering.
- Candidates (named before scoring): `baseline` (class prior), `logreg` (scaled + one-hot site), `gbm` (HistGradientBoosting, small grid on learning rate/depth).
- Metric: average precision (PR-AUC), since failures are ~12%. The decision threshold is chosen on out-of-fold CV predictions to maximise F1.

In [ ]:
import numpy as np, pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import StandardScaler, OneHotEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import HistGradientBoostingClassifier
from sklearn.dummy import DummyClassifier
from sklearn.metrics import average_precision_score, roc_auc_score, f1_score, precision_score, recall_score
from sklearn.base import clone

df = pd.read_csv('data.csv', parse_dates=['inspected_at']).sort_values('inspected_at').reset_index(drop=True)
print(df.shape, df.asset_tag.is_unique, df.failed_90d.mean())
print(df.groupby('parts_replaced').failed_90d.agg(['mean','size']))

## 1. Hold out the test set (latest 20% of rows) before any modelling

In [ ]:
cut = int(len(df)*0.8)
train, test = df.iloc[:cut].copy(), df.iloc[cut:].copy()
print(train.inspected_at.max(), '<', test.inspected_at.min())
print('train rate', train.failed_90d.mean(), 'test rate', test.failed_90d.mean())
NUM = ['vibration_rms','bearing_temp_c','ambient_temp_c','meter_hours','load_pct']
CAT = ['site']
FEATURES = NUM + CAT   # parts_replaced (post-inspection), asset_tag (id), inspected_at excluded
X_tr, y_tr = train[FEATURES], train.failed_90d.values
X_te, y_te = test[FEATURES], test.failed_90d.values

## 2-4. Candidate models (all named up front) and time-ordered CV on training rows only
Preprocessing lives inside each pipeline, so it is fitted per fold.

In [ ]:
pre = lambda: ColumnTransformer([('n', StandardScaler(), NUM), ('c', OneHotEncoder(handle_unknown='ignore'), CAT)])
cands = {'baseline': Pipeline([('p', pre()), ('m', DummyClassifier(strategy='prior'))]),
         'logreg':   Pipeline([('p', pre()), ('m', LogisticRegression(max_iter=1000))])}
for lr in (0.03, 0.1):
    for d in (2, 3, 5):
        cands[f'gbm_lr{lr}_d{d}'] = Pipeline([('p', pre()), ('m', HistGradientBoostingClassifier(learning_rate=lr, max_depth=d, max_iter=200, random_state=0))])
print(list(cands))
tscv = TimeSeriesSplit(n_splits=5)
oof, rows = {}, []
for name, pipe in cands.items():
    p = np.full(len(train), np.nan); aps=[]
    for a, b in tscv.split(X_tr):
        m = clone(pipe).fit(X_tr.iloc[a], y_tr[a])
        p[b] = m.predict_proba(X_tr.iloc[b])[:,1]
        aps.append(average_precision_score(y_tr[b], p[b]))
    oof[name] = p; rows.append((name, np.mean(aps), np.std(aps)))
cv = pd.DataFrame(rows, columns=['model','cv_ap_mean','cv_ap_std']).sort_values('cv_ap_mean', ascending=False)
cv

## 5. Choose model and threshold on CV only

In [ ]:
best = cv.iloc[0].model
mask = ~np.isnan(oof[best])
ths = np.linspace(0.02, 0.8, 79)
f1s = [f1_score(y_tr[mask], oof[best][mask] >= t) for t in ths]
THRESH = float(ths[int(np.argmax(f1s))])
print('chosen model:', best, '| threshold:', round(THRESH,3), '| CV F1:', round(max(f1s),3))

**Choice (recorded before touching the test set):** the model and threshold printed above, features = the six listed in `FEATURES`.

## 6. Refit on all training rows, score the test set once

In [ ]:
final = clone(cands[best]).fit(X_tr, y_tr)
p = final.predict_proba(X_te)[:,1]; pred = p >= THRESH
print('Test AP     :', round(average_precision_score(y_te, p),3), '(prevalence', round(y_te.mean(),3), ')')
print('Test ROC AUC:', round(roc_auc_score(y_te, p),3))
print('Test F1/precision/recall @ threshold:', round(f1_score(y_te,pred),3), round(precision_score(y_te,pred),3), round(recall_score(y_te,pred),3))

## Notes and caveats
- The headline is the single test-set score above; nothing was tuned on it.
- Failure rate drifts strongly over time (about 27% in 2024 Q2-Q3 vs 3% in 2025 Q4), so the test prevalence differs from training. Calibration/threshold may not transfer; part of the late-period drop could be label censoring if failures after the data end are unobserved.
- Had `parts_replaced` been used it would have given near-perfect but meaningless scores, since it is only known after the outcome-relevant work.